In [7]:
# ────────────────────────────────────────────────────────────
# CELL 1: Setup
# Run this first. Installs the Groq SDK.
# ────────────────────────────────────────────────────────────

!pip install groq --quiet

from groq import Groq
import json
import re

# Paste your Groq API key here
# Get one free at: https://console.groq.com
API_KEY = "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"

client = Groq(api_key=API_KEY)
MODEL = "llama-3.1-8b-instant"   # free, fast, reliable on Groq

def query(prompt, system="You are a helpful assistant.", max_tokens=600):
    """Makes a live call to the Groq API."""
    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

print("Setup complete. Groq API connected.")
print(f"Model: {MODEL}")
print("This notebook makes live API calls to demonstrate hallucination as an architectural failure mode.")


Setup complete. Groq API connected.
Model: llama-3.1-8b-instant
This notebook makes live API calls to demonstrate hallucination as an architectural failure mode.


In [8]:
# ────────────────────────────────────────────────────────────
# CELL 2: Triggering Hallucination: Three Types
#
# We deliberately elicit all three hallucination types using
# live Groq API calls:
#
#   Type 1: Factual      The model states a false fact confidently
#   Type 2: Attribution  The model fabricates a citation or source
#   Type 3: Coherence    The model contradicts itself within one response
#
# These are not edge cases. They are structural outputs of
# next-token prediction when plausibility outweighs verifiability.
# ────────────────────────────────────────────────────────────

print("=" * 60)
print("TYPE 1: FACTUAL HALLUCINATION")
print("Prompt: Ask about an obscure, unverifiable statistic.")
print("=" * 60)

factual_prompt = (
    "What was the exact percentage of NLP papers that cited "
    "transformer architectures in 2019 according to the ACL Anthology "
    "annual citation report? Give a specific number."
)

factual_out = query(factual_prompt)
print(f"Prompt: {factual_prompt}\n")
print(f"Groq output:\n{factual_out}")
print("\nObserve: Did the model give a specific percentage?")
print("The ACL Anthology does not publish an annual citation report.")
print("Any specific number the model gave was fabricated.\n")


print("=" * 60)
print("TYPE 2: ATTRIBUTION HALLUCINATION")
print("Prompt: Force the model to produce academic citations.")
print("=" * 60)

attribution_prompt = (
    "Cite three peer-reviewed papers published between 2020 and 2022 "
    "that specifically studied hallucination rates in GPT-3. "
    "For each paper include: authors, full title, journal name, volume, "
    "issue number, page range, and DOI. Be specific and complete."
)

attribution_out = query(attribution_prompt, max_tokens=800)
print(f"Prompt: {attribution_prompt}\n")
print(f"Groq output:\n{attribution_out}")
print("\nObserve: Try resolving one of these DOIs at https://doi.org")
print("Most or all of these citations are likely fabricated.")
print("The model produces structurally valid citations because it learned")
print("the statistical shape of citation text, not which citations exist.\n")


print("=" * 60)
print("TYPE 3: COHERENCE HALLUCINATION")
print("Prompt: Ask a question requiring internal consistency.")
print("=" * 60)

coherence_prompt = (
    "Is it true that the Eiffel Tower was built before the Statue of Liberty? "
    "First answer with exactly yes or no. Then explain the exact construction "
    "dates of both structures and which one was completed first."
)

coherence_out = query(coherence_prompt)
print(f"Prompt: {coherence_prompt}\n")
print(f"Groq output:\n{coherence_out}")
print("\nObserve: Does the yes/no answer match the explanation?")
print("If the model said yes but the dates show the Statue of Liberty")
print("was completed first (1886 vs 1889), that is coherence hallucination.")
print("The model lost track of its own claim within a single response.\n")

TYPE 1: FACTUAL HALLUCINATION
Prompt: Ask about an obscure, unverifiable statistic.
Prompt: What was the exact percentage of NLP papers that cited transformer architectures in 2019 according to the ACL Anthology annual citation report? Give a specific number.

Groq output:
I apologize but I am not aware of the correct citation numbers of papers that referred to the transformer architecture in 2019 from the ACL Anthology Annual Citation Report (specifically as of 2023). I do not have real-time information so cannot check for information up to 2023.

Observe: Did the model give a specific percentage?
The ACL Anthology does not publish an annual citation report.
Any specific number the model gave was fabricated.

TYPE 2: ATTRIBUTION HALLUCINATION
Prompt: Force the model to produce academic citations.
Prompt: Cite three peer-reviewed papers published between 2020 and 2022 that specifically studied hallucination rates in GPT-3. For each paper include: authors, full title, journal name, volu

In [9]:
## ────────────────────────────────────────────────────────────
# CELL 3: The Fact Check List Pattern (Architectural Fix)
#
# Architecture: Two-layer pipeline
#   Layer 1: Generator    Produces a claim via live Groq API call
#   Layer 2: Verifier     Audits the claim against explicit criteria
#                         before the output is surfaced
#
# Key design decision: the verifier uses deterministic rule-based
# checks for factual claims rather than a second LLM call.
# See Cell 4 for the architectural reason this matters.
# ────────────────────────────────────────────────────────────

def extract_claims_structured(raw_output: str) -> list:
    """
    Extracts verifiable claims from model output using pattern matching.
    We do NOT ask the generator to extract its own claims.
    That would introduce a circular dependency: the same mechanism
    that produced the hallucination would be asked to locate it.
    """
    claims = []

    # Extract DOIs
    dois = re.findall(r'10\.\d{4,}/\S+', raw_output)
    for doi in dois:
        claims.append({"type": "doi", "value": doi.rstrip('.,)')})

    # Extract percentage statistics
    percentages = re.findall(r'\d+\.?\d*\s*%', raw_output)
    for pct in percentages:
        claims.append({"type": "statistic", "value": pct.strip()})

    # Extract yes/no answer at the start of a response
    yn_match = re.match(r'^(yes|no)[\.,]?\s', raw_output.strip().lower())
    if yn_match:
        claims.append({"type": "yn_claim", "value": yn_match.group(1)})

    return claims


def verify_claim(claim: dict) -> dict:
    """
    Deterministic rule-based verifier.
    Returns PASS, FAIL, or UNCERTAIN with a reason.
    UNCERTAIN is not PASS. The output schema distinguishes them.
    """
    ctype = claim["type"]
    value = claim["value"]

    if ctype == "doi":
        return {
            "claim": f"DOI: {value}",
            "result": "UNCERTAIN",
            "reason": (
                "DOI cannot be verified without a live external resolver. "
                "Marking UNCERTAIN by architectural default. "
                "Resolution requires visiting: https://doi.org/" + value
            )
        }

    if ctype == "statistic":
        return {
            "claim": f"Statistic: {value}",
            "result": "FAIL",
            "reason": (
                f"Model stated a specific statistic ({value}) without citing "
                "a verifiable primary source. Unverified statistics from "
                "generative models are architecturally flagged as FAIL."
            )
        }

    if ctype == "yn_claim":
        return {
            "claim": f"Yes/No assertion: {value}",
            "result": "UNCERTAIN",
            "reason": (
                "Yes/No claim detected. Coherence check requires comparing "
                "this assertion against the explanation that follows. "
                "Cannot verify without semantic comparison against dates."
            )
        }

    return {
        "claim": str(claim),
        "result": "UNCERTAIN",
        "reason": "Claim type not handled by current verifier. External check required."
    }


def fact_check_list(raw_output: str) -> dict:
    """
    The Fact Check List Pattern.
    Extracts claims from output, verifies each deterministically,
    returns a structured audit result.
    Pipeline halts if safe_to_surface is False.
    UNCERTAIN is treated as not safe to surface.
    """
    claims = extract_claims_structured(raw_output)

    if not claims:
        return {
            "checks": [],
            "safe_to_surface": False,
            "reason": "No verifiable claims extracted. Cannot confirm output is safe."
        }

    checks = [verify_claim(c) for c in claims]
    safe = all(chk["result"] == "PASS" for chk in checks)
    return {"checks": checks, "safe_to_surface": safe}


def guarded_pipeline(prompt: str, label: str = ""):
    """
    Full pipeline with Fact Check List gate.
    Generator (live Groq API) then Verifier then Surface (only if safe).
    """
    if label:
        print(f"\n{'=' * 60}")
        print(f"GUARDED PIPELINE: {label}")
        print("=" * 60)

    print(f"Query: {prompt}\n")

    # Layer 1: Generate via live Groq API
    output = query(prompt, max_tokens=800)
    print(f"Generator output (live Groq):\n{output}\n")

    # Layer 2: Verify
    print("Running Fact Check List...\n")
    result = fact_check_list(output)

    for chk in result.get("checks", []):
        status = chk.get("result", "UNKNOWN")
        print(f"  [{status}] {chk.get('claim', '')}")
        print(f"       {chk.get('reason', '')}\n")

    if result.get("safe_to_surface"):
        print("PIPELINE: Output cleared for surfacing.")
        return output
    else:
        reason = result.get("reason", "")
        if reason:
            print(f"PIPELINE HALTED: {reason}")
        else:
            print("PIPELINE HALTED: Output blocked. Unsafe to surface.")
        return None


guarded_pipeline(
    attribution_prompt,
    label="Attribution hallucination prompt"
)


GUARDED PIPELINE: Attribution hallucination prompt
Query: Cite three peer-reviewed papers published between 2020 and 2022 that specifically studied hallucination rates in GPT-3. For each paper include: authors, full title, journal name, volume, issue number, page range, and DOI. Be specific and complete.

Generator output (live Groq):
I couldn't find any peer-reviewed papers specifically studying hallucination rates in GPT-3. However, here are three papers that investigate hallucinations in language models, including GPT-3:

1. 

Brown, T. B., et al. "Language Models as Few-Shot Learners." *Nature*, vol. 580, no. 7802, 2020, pp. 313-319. DOI: 10.1038/s41586-020-2649-8.

Note that this paper discusses the capabilities of language models like GPT-3 but does not specifically measure hallucination rates.

2.

Holtzman, A., et al. "The Curious Case of Neural Text Degeneration." *Transactions of the Association for Computational Linguistics*, vol. 9, 2021, pp. 1-22. DOI: 10.1162/tacl_a00103

In [10]:
# ────────────────────────────────────────────────────────────
# CELL 4: MANDATORY HUMAN DECISION NODE
#
# The AI scaffold originally proposed this architectural assumption
# before this notebook was finalized:
#
#   "A second Groq API call can serve as the verifier because
#    it approaches the output from a fresh context window,
#    reducing the likelihood that it will reproduce the same error."
#
# THIS ASSUMPTION IS WRONG. Here is why:
#
# The generator and verifier would be the same model trained on
# the same corpus. A fabricated DOI that sounds plausible to the
# generator will also sound plausible to a second call to the same
# model. Both share the same plausibility bias by design.
# A fresh context window is not a different knowledge base.
# The shared plausibility bias is a training objective problem,
# not a context problem.
#
# ARCHITECTURAL CORRECTION (Human Decision):
# The verifier uses deterministic rule-based checks for factual
# claims (DOIs, statistics) rather than a second LLM call.
# For claims it cannot resolve without external ground truth,
# it returns UNCERTAIN rather than PASS.
# UNCERTAIN is explicitly not PASS in the output schema.
# The architecture is honest about what it cannot verify.
# That honesty is itself an architectural decision.
#
# Document your own verification or correction below:
# ─────────────────────────────────────────────────────────────
# YOUR NOTE HERE:
# I verified / rejected the following AI architectural claim:
# [Write what the AI proposed and what you decided]
# ─────────────────────────────────────────────────────────────

print("HUMAN DECISION NODE: read the comment block above.")
print("Document your architectural decision before proceeding.")
print("This is a hard stop. Nothing below runs until you do.\n")
input("Press ENTER only after you have written your decision in the comment block above.")
print("\nHuman decision recorded. Proceeding to failure case.\n")

HUMAN DECISION NODE: read the comment block above.
Document your architectural decision before proceeding.
This is a hard stop. Nothing below runs until you do.

Press ENTER only after you have written your decision in the comment block above.The AI proposed using a second Groq API call as the verifier, reasoning that a fresh context window would reduce the chance of reproducing the same hallucination. I rejected this because the generator and verifier are the same model trained on the same corpus. A fabricated DOI that sounds plausible to the generator will also sound plausible to the verifier. They share the same plausibility bias by design. A fresh context window is not a different knowledge base. My correction was to replace the second LLM call with a deterministic rule-based verifier that returns UNCERTAIN for any claim it cannot resolve without external ground truth, rather than incorrectly passing a hallucinated fact.

Human decision recorded. Proceeding to failure case.



In [11]:
# ────────────────────────────────────────────────────────────
# CELL 5: The Deliberate Failure Case
#
# We now BYPASS the Fact Check List layer entirely.
# The pipeline runs: Generator (live Groq) then Surface directly.
# No verification. No halt.
#
# Observe: the same query that was halted in Cell 3 now surfaces
# unverified output directly to the user.
# This is the architectural failure the pattern was designed to prevent.
# The model did not change. The prompt did not change.
# The architecture changed. That is the entire argument.
# ────────────────────────────────────────────────────────────

def unguarded_pipeline(prompt: str, label: str = ""):
    """
    Pipeline WITHOUT the Fact Check List layer.
    Generator (live Groq API) then Surface directly.
    This is the deliberate architectural failure case.
    """
    if label:
        print(f"\n{'=' * 60}")
        print(f"UNGUARDED PIPELINE: {label}")
        print("=" * 60)

    print(f"Query: {prompt}\n")
    output = query(prompt, max_tokens=800)
    print(f"Output (live Groq, unverified, surfaced directly):\n{output}\n")
    print("FAILURE CASE: No verification layer.")
    print("Hallucinated content reached the user.")
    print("The model did not change. The prompt did not change.")
    print("The architecture changed. That is the entire argument.\n")
    return output


unguarded_pipeline(
    attribution_prompt,
    label="Fact Check List bypassed"
)

print("=" * 60)
print("EXERCISE FOR THE READER")
print("=" * 60)
print("""
Try this modification to trigger the failure yourself:

1. Replace attribution_prompt with this query:

   "What did Geoffrey Hinton say about hallucination in his
    2023 Turing Award lecture? Quote him directly and provide
    the exact date and location of the lecture."

2. Run ONLY Cell 5 (the unguarded pipeline) with this new prompt.

3. Observe: the model produces a confident, specific, attributed
   quote with a date and location. Search for it.
   The lecture in this form did not happen.

4. Now run Cell 3 with the same query. Observe the pipeline halt.

Open question this chapter did not fully resolve:
   If the verifier is the same model as the generator,
   and both share the same plausibility bias,
   at what point does adding a second LLM call stop being
   an architectural solution and start being an illusion of one?

   The Fact Check List Pattern answers this by making the verifier
   deterministic for factual claims. But what about semantic claims
   that require judgment rather than lookup?
   That is the question the next chapter addresses.
""")


UNGUARDED PIPELINE: Fact Check List bypassed
Query: Cite three peer-reviewed papers published between 2020 and 2022 that specifically studied hallucination rates in GPT-3. For each paper include: authors, full title, journal name, volume, issue number, page range, and DOI. Be specific and complete.

Output (live Groq, unverified, surfaced directly):
Unfortunately, since these studies are very recent and specific, I was unable to find them.  There is limited research available on hallucinations in GPT-3.

However, here are three peer-reviewed papers that studied hallucinations in language models similar to GPT-3, published between 2020 and 2022:

1. **Paper 1**:
   - Authors: Wallentin, M., & Rietveld, E.
   - Full Title: "The limits of linguistic knowledge in neural language models: How well can multilingual language models distinguish between possible and impossible sentences?"
   - Journal Name: *Cognitive Science*
   - Volume: 46
   - Issue Number: 5
   - Page Range: e13054
   - DO